# VoyageRL 实验笔记
从项目根目录启动 Jupyter。数据示例使用本地归档路径；真实 API 调用前请配置 `.env`。训练日志由你自己的训练生成。

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), Path.cwd().parent)
     if (p / "run_tool_loop_infer.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from the project root or notebooks directory.")
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


# 第一级：规则清洗

In [ ]:
import os
import re
import json
from langdetect import detect

save_dir = "saved/train_expansion"
files = os.listdir(save_dir)
tool_calls_stat = {}
droped_count = 0
total_count = 0
train_data = []

stat_turns = []
for filename in files:
    if filename.endswith(".json"):
        info = json.load(open(os.path.join(save_dir, filename)))
        question = info["question"]
        msg = info["messages"][-1]["content"]
        stats = info["stats"]
        turn = stats.pop("turns", 0) # 对话轮次
        num_calls = sum(stats.values()) # 工具调用的总次数
        
        if turn > 13 or num_calls == 0: # 调用超过13次或者没有调用工具
            droped_count += 1
            continue
            
        elif "exceeds the limit" in msg or "reached the maximum" in msg: # 超过最大对话轮次/上下文最大长度
            droped_count += 1
            continue
            
        elif  "没有找到" in msg:   # 答案里面"没有找到"（query不合理或者query缺少信息）
            droped_count += 1  
            continue
            
        elif  detect(msg) == "en":  # 答案语言是英文
            droped_count += 1
            continue
            
        elif len(msg) < 100:     # 答案太简短
            droped_count += 1
            continue

        # 记录工具调用情况
        for name, value in stats.items():
            if name not in tool_calls_stat:
                tool_calls_stat[name] = 0
            tool_calls_stat[name] += 1

        new_info = {
            "qid": filename.split(".")[0],
            "question": info["question"],
            "answer": json.dumps(info["messages"], ensure_ascii=False)
        }

        train_data.append(new_info)
        stat_turns.append(turn)
            
        total_count += 1

# 统计工具的调用次数
for k, v in tool_calls_stat.items():
    print(k, v)
print()
print(f"total count: {total_count}, dropped count: {droped_count}")

In [ ]:
import numpy as np

for k in [50, 60, 70, 80, 90, 95]:
    print(f"{k}分位点: {np.percentile(stat_turns, k)}")

In [ ]:
import os
import re
import json
from langdetect import detect

save_dir = "saved/test"
files = os.listdir(save_dir)
tool_calls_stat = {}
droped_count = 0
total_count = 0
test_data = []
for filename in files:
    if filename.endswith(".json"):
        info = json.load(open(os.path.join(save_dir, filename)))
        question = info["question"]
        msg = info["messages"][-1]["content"]
        stats = info["stats"]
        turn = stats.pop("turns", 0)
        num_calls = sum(stats.values())
        if turn > 13 or num_calls == 0: # 调用超过13次或者没有调用工具
            droped_count += 1
            continue
        elif "exceeds the limit" in msg or "reached the maximum" in msg: # 超过最大对话轮次
            droped_count += 1
            continue
        elif  "没有找到" in msg:   # 答案里面"没有找到"（query不合理或者query缺少信息）
            droped_count += 1  
            continue
        elif  detect(msg) == "en":   # 答案语言全是英文
            droped_count += 1
            continue
        elif len(msg) < 100:     # 答案太简短
            droped_count += 1
            continue

        # 记录工具调用情况
        for name, value in stats.items():
            if name not in tool_calls_stat:
                tool_calls_stat[name] = 0
            tool_calls_stat[name] += 1

        new_info = {
            "qid": filename.split(".")[0],
            "question": info["question"],
            "answer": json.dumps(info["messages"], ensure_ascii=False)
        }
            
        total_count += 1
        test_data.append(new_info)

# 统计工具的调用次数
for k, v in tool_calls_stat.items():
    print(k, v)
print()
print(f"total count: {total_count}, dropped count: {droped_count}")

In [ ]:
train_data[0]

## 格式化messages并保存

In [ ]:
save_dir = "data/local/clean/"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

iterators = [
    [train_data, open(save_dir + "train.jsonl", "w")],
    [test_data, open(save_dir + "test.jsonl", "w")],
]
all_data = []
for data, handler in iterators:
    for item in data:
        messages = json.loads(item["answer"])
        conversations = []
        for message in messages:
            if message["role"] == "system":
                message["content"] = message["content"].replace("最大可调用50轮工具", "最大可调用13轮工具")
                info = {
                    "role": "system",
                    "content": message["content"]
                }
            elif message["role"] == "user":
                info = {
                    "role": "user",
                    "content": message["content"]
                }
            elif message["role"] == "assistant":
                # 处理tool calls和answer
                reasoning_body = message["reasoning_details"]
                reasoning_content = "\n".join([reasoning_body[t]["text"] for t in range(len(reasoning_body))])
                if reasoning_content.strip():
                    reasoning_content = f"<think>\n{reasoning_content}\n</think>\n\n"
                else:
                    reasoning_content = ""
                if message["tool_calls"]:
                    content = json.dumps(message["tool_calls"], ensure_ascii=False)
                    info = {
                        "role": "assistant", 
                        "content": f"{reasoning_content}<tool_call>\n{content}\n</tool_call>"
                    }
                elif message["content"].strip():
                    # 处理答案
                    content = message["content"]
                    info = {
                        "role": "assistant", 
                        "content": f"{reasoning_content}{content}"
                    }   
                
            elif message["role"] == "tool":
                # 处理工具返回
                info = {
                    "role": "user",
                    "content": f"<tool_response>\n{message['content']}\n</tool_response>"
                }
            conversations.append(info)
        body = {
            "id": item["qid"],
            "conversations": conversations
        }
        all_data.append(body)
        handler.write(json.dumps(body, ensure_ascii=False) + "\n")

In [ ]:
import random
random.choice(all_data)["conversations"]